# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's look at the available record sets, their `@id`s, and contained fields/columns.

In [ ]:
record_sets = metadata.recordSet
print('Available record sets:')
record_set_ids = []
for record_set in record_sets:
    print(f"- RecordSet: {record_set['@id']}, name: {record_set.get('name', '(no name)')}")
    record_set_ids.append(record_set['@id'])

    fields = record_set.get('field', [])
    print('  Fields:')
    for field in fields:
        print(f"    - Field @id: {field['@id']}, name: {field.get('name', '(no name)')}, dataType: {field.get('dataType', '(not specified)')}")
    
    columns = record_set.get('column', [])
    if columns:
        print('  Columns:')
        for col in columns:
            print(f"    - Column @id: {col['@id']}, name: {col.get('name', '(no name)')}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

We will use the record set `@id`s and fields identified above.

In [ ]:
dataframes = {}
# Store fields by record set as a helper
record_set_fields_dict = {}

print('Extracting records to pandas DataFrames:')
for record_set in record_sets:
    recset_id = record_set['@id']
    print(f"RecordSet @id: {recset_id}")
    records = list(dataset.records(record_set=recset_id))
    df = pd.DataFrame(records)
    dataframes[recset_id] = df
    print(f"Columns: {df.columns.tolist()}")
    if len(df) > 0:
        print(df.head())

    # Store field names and IDs for later
    fields = record_set.get('field', [])
    record_set_fields_dict[recset_id] = [(f['@id'], f.get('name', f['@id']), f.get('dataType', None)) for f in fields]

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records, normalizing numeric fields, and grouping data.

We select a record set and numeric field for detailed analysis.

In [ ]:
# Select the first record set for analysis
selected_record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes[selected_record_set_id]

# Identify numeric fields (those with dataType matching 'Float', 'Integer', or 'Number')
numeric_fields = [f[1] for f in record_set_fields_dict[selected_record_set_id] if f[2] in ('Float', 'Integer', 'Number')]
print(f"Numeric fields in record set {selected_record_set_id}: {numeric_fields}")

# Pick the first numeric field for demonstration
if numeric_fields:
    numeric_field = numeric_fields[0]

    threshold = 10
    if numeric_field in df.columns:
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by another field in the same record set
        possible_group_fields = [f[1] for f in record_set_fields_dict[selected_record_set_id] if f[1] != numeric_field]
        group_field = None
        for gf in possible_group_fields:
            if gf in df.columns and df[gf].dtype == 'O':
                group_field = gf
                break

        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())

## 5. Visualization
Visualize data distributions and relationships between fields in the dataset.

In [ ]:
# Example: Plot histogram for the selected numeric field
if numeric_fields and numeric_field in df.columns:
    plt.figure(figsize=(6, 4))
    sns.histplot(df[numeric_field].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

# If available, plot numeric_field vs group_field
if group_field and numeric_field in df.columns and group_field in df.columns:
    plt.figure(figsize=(6, 4))
    sns.boxplot(x=df[group_field], y=df[numeric_field])
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset documents clinical features, hormone levels, imaging findings, and genotype information for three female patients with non-classical 21-hydroxylase deficiency.
- Record sets and fields are referenced by their unique `@id`s for traceability.
- Numeric values (e.g., age or hormone levels) can be explored for outliers and normalized for comparative analysis.
- Grouping and visualization can highlight phenotype heterogeneity by clinical attributes.

Further analysis may combine information across record sets to enable nuanced genotype–phenotype correlation.